# Approximation Error Measurement During Generation

**Purpose:** Measure the L2 approximation error of sparse attention methods during actual text generation (not just on cached vectors).

**What it does:**
- Runs text generation with both full attention (ground truth) and sparse attention (Jungle or MagicPIG)
- Computes token-by-token L2 error: `||output_sparse - output_full||_2`
- Tracks error statistics across generation steps
- Visualizes error evolution during decoding

**Key metrics:**
- Per-token approximation error
- Average error across full generation sequence
- Error distribution and outliers

**Purpose:** Validates that sparse methods maintain low approximation error during actual inference, not just on static cached data.

In [1]:
import os
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"  # Set your HF token or use `huggingface-cli login`

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
import math
import collections
import time
import gc
import numpy as np

# ==========================================
# Part 1: Utility Functions
# ==========================================

def cleanup_memory():
    """Aggressively clears memory to prevent OOM loops."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
    q_f32 = q.float()
    cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
    sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
    q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
    return q_embed.to(q.dtype)

def manual_rmsnorm(hidden_states, weight, variance_epsilon):
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.pow(2).mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return weight * hidden_states.to(input_dtype)

def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = cumulative_probs > top_p
    mask[:, :, 1:] = mask[:, :, :-1].clone()
    mask[:, :, 0] = False
    sorted_probs.masked_fill_(mask, 0.0)
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
    sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
    final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
    return final_indices.squeeze(-1)

# ==========================================
# Part 2: LSH Implementation (Legacy MagicPIG)
# ==========================================

class LSH:
    def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
        self.K = K
        self.L = L
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.batch_size = batch_size
        self.max_length = max_length
        self.num_attention_groups = num_heads // num_kv_heads
        self.device = device
        self.tables = []
        for _ in range(num_layers):
            req_tables = []
            for _ in range(batch_size):
                head_tables = []
                for _ in range(num_kv_heads):
                    l_tables = [collections.defaultdict(list) for _ in range(L)]
                    head_tables.append(l_tables)
                req_tables.append(head_tables)
            self.tables.append(req_tables)

    def clear(self):
        for layer_idx in range(self.num_layers):
            for req_id in range(self.batch_size):
                for head_idx in range(self.num_kv_heads):
                    for l in range(self.L):
                        self.tables[layer_idx][req_id][head_idx][l].clear()

    def fill(self, layer_id, request_id, hash_codes, indices):
        hc = hash_codes.cpu()
        idx = indices.cpu()
        num_kv, L, seq_len = hc.shape
        for h in range(num_kv):
            for l in range(L):
                table_dict = self.tables[layer_id][request_id][h][l]
                current_hashes = hc[h, l].tolist()
                current_indices = idx.tolist()
                for i, val in enumerate(current_hashes):
                    table_dict[val].append(current_indices[i])

    def batch_retrieve(self, layer_id, query_hash_codes):
        B_H, L = query_hash_codes.shape
        query_hash_codes = query_hash_codes.cpu()
        results = []
        for i in range(B_H):
            req_id = i // self.num_heads
            local_head_id = i % self.num_heads
            kv_head_id = local_head_id // self.num_attention_groups
            counts = collections.defaultdict(int)
            for l in range(L):
                val = query_hash_codes[i, l].item()
                bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
                for idx in bucket:
                    counts[idx] += 1
            candidates = [idx for idx, count in counts.items() if count >= 2]
            if not candidates:
                t_cand = torch.empty(0, dtype=torch.long, device=self.device)
            else:
                t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
            results.append(t_cand)
        return results

# ==========================================
# Part 3: Sparse Attention Math (SNIS Kernels)
# ==========================================

def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
    if k.shape[0] == 0:
        return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
    score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
    q_norm_f = q.float().norm(p=2)
    k_norm_f = k_norm.float()
    denom = q_norm_f * k_norm_f
    cos_theta = score_f / (denom + 1e-6)
    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
    theta = torch.acos(cos_theta)
    prob = 1.0 - theta / math.pi
    p = prob.pow(K)
    q_prob = 1.0 - p
    w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
    log_w = torch.log(w + 1e-4)
    if is_exact is not None:
        log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
    score_f = score_f / math.sqrt(head_dim) - log_w
    attn_probs = torch.softmax(score_f, dim=0)
    return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
    if k.shape[0] == 0:
        return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
    score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
    q_norm_f = q.float().norm(p=2)
    k_norm_f = k_norm.float()
    denom = q_norm_f * k_norm_f
    cos_theta = score_f / (denom + 1e-6)
    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
    theta = torch.acos(cos_theta)
    p_base = 1.0 - theta / math.pi

    p_collision = p_base.pow(retrieval_depths.float())
    w = 1.0 - (1.0 - p_collision).pow(L)
    log_w = torch.log(torch.clamp(w, min=1e-10))

    if is_exact is not None:
        log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

    score_f = score_f / math.sqrt(head_dim) - log_w
    attn_probs = torch.softmax(score_f, dim=0)
    return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# ==========================================
# Part 4: Logging Infrastructure
# ==========================================

class AttentionLogger:
    def __init__(self):
        self.reset()

    def reset(self):
        self.stats = collections.defaultdict(list)

    def log(self, key, value):
        self.stats[key].append(value)

    def get_aggregated_stats(self):
        """Returns dictionary of raw means for experimentation."""
        res = {}
        if not self.stats["total_tokens"]:
            return res

        total = np.mean(self.stats["total_tokens"])
        sparse = np.mean(self.stats["sparse_tokens"])
        res['avg_total_tokens'] = total
        res['avg_sparse_tokens'] = sparse
        res['sparsity_ratio'] = sparse / total if total > 0 else 0
        return res

# ==========================================
# Part 5: The "Server" (Memory Manager)
# ==========================================

class LSHSparseAttnServer:
    def __init__(self, config, K=10, L=150, batch_size=1,
                 num_sink_tokens=4, num_local_tokens=64,
                 max_length=8192, dense_layers=[0, 16, 32],
                 device='cuda:0', dtype=torch.bfloat16, verbose=False,
                 use_jungle=False, jg_budget=0.05):

        self.approx_errors = []
        self.config = config
        self.K = K
        self.L = L
        self.batch_size = batch_size
        self.num_sink_tokens = num_sink_tokens
        self.num_local_tokens = num_local_tokens
        self.dense_layers = set(dense_layers)
        self.device = device
        self.dtype = dtype
        self.verbose = verbose
        self.use_jungle = use_jungle

        # Logging
        self.logger = AttentionLogger()
        self.log_interval = 2
        self.logging_layer = 15

        # Strict Fairness Config
        self.jg_K_max = K
        self.jg_L = L
        self.jg_budget = jg_budget
        self.jg_min_depth = 1

        self.num_layers = config.num_hidden_layers
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // self.num_heads
        self.num_attention_groups = self.num_heads // self.num_kv_heads

        self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.current_len = [0] * batch_size

        self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
        self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
        self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

        if self.use_jungle:
            self.jg_projs = torch.randn(self.head_dim, self.jg_L * self.jg_K_max, device=device, dtype=dtype)
            self.jg_hash_cache = collections.defaultdict(dict)

        self.sparse_boundaries = {}

    def clear(self):
        self.lsh.clear()
        self.logger.reset()
        self.approx_errors = []
        self.step_counter = 0
        for i in range(self.batch_size):
            self.current_len[i] = 0
        for l in range(self.num_layers):
            self.k_cache[l].zero_()
            self.v_cache[l].zero_()
            self.avg_k_cache[l].zero_()
        self.sparse_boundaries = {}
        if self.use_jungle:
            self.jg_hash_cache.clear()

    def step(self):
        self.step_counter += 1
        for i in range(self.batch_size):
            self.current_len[i] += 1

    def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
        seq_len = key_states.shape[0]
        end_pos = start_pos + seq_len
        self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
        self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
        if end_pos > self.current_len[request_id]:
            self.current_len[request_id] = end_pos

        if layer_idx not in self.dense_layers:
            idx_start = max(start_pos, self.num_sink_tokens)
            idx_end = end_pos - self.num_local_tokens
            self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

            if idx_end > idx_start:
                keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
                avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
                self.avg_k_cache[layer_idx][request_id] = avg_k
                centered_keys = keys_to_index - avg_k

                if self.use_jungle:
                    jg_proj = torch.matmul(centered_keys, self.jg_projs)
                    jg_bits = (jg_proj > 0).float()
                    n_sparse = jg_bits.shape[1]
                    jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
                    if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
                    self.jg_hash_cache[layer_idx][request_id] = jg_bits
                else:
                    projected = torch.matmul(centered_keys, self.hash_func)
                    bits = (projected > 0).float()
                    bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
                    buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
                    indices = torch.arange(idx_start, idx_end, device=self.device)
                    self.lsh.fill(layer_idx, request_id, buckets, indices)

    def decode(self, query_states, key_states, value_states, layer_idx):
        bsz, n_heads, q_len, dim = query_states.shape
        for req_id in range(bsz):
            curr_len = self.current_len[req_id]
            self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
            self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

        hidden_states_list = []

        # Init counters
        log_sink = 0
        log_local = 0
        log_sparse = 0
        log_depths = []

        for req_id in range(bsz):
            q_heads = query_states[req_id, :, 0, :]
            curr_len = self.current_len[req_id]

            if layer_idx in self.dense_layers:
                k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
                v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
                k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
                v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
                scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
                attn = torch.softmax(scores, dim=-1)
                out = torch.matmul(attn, v.float()).squeeze(1)
                hidden_states_list.append(out.to(self.dtype))
            else:
                head_outputs = []
                norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

                if self.use_jungle:
                    jg_q_proj = torch.matmul(norm_q, self.jg_projs)
                    jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
                else:
                    projected = torch.matmul(norm_q, self.hash_func)
                    bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
                    q_buckets = torch.matmul(bits, self.binary_pack).long()
                    idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

                for h in range(self.num_heads):
                    kv_head = h // self.num_attention_groups
                    sparse_boundary = self.sparse_boundaries.get(req_id, 0)

                    sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
                    local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

                    sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
                    retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

                    if self.use_jungle:
                        if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
                            k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
                            q_bits_h = jg_q_bits[h]
                            match = (k_bits == q_bits_h.unsqueeze(0)).int()
                            depths = match.cumprod(dim=-1).sum(dim=-1)
                            max_d, _ = depths.max(dim=-1)

                            N_sparse = max_d.shape[0]
                            budget = int(N_sparse * self.jg_budget)
                            if budget > 0:
                                sorted_d, sorted_idx = torch.sort(max_d, descending=True)
                                valid_mask = sorted_d >= self.jg_min_depth
                                valid_idx = sorted_idx[valid_mask]
                                take = min(budget, valid_idx.numel())
                                if take > 0:
                                    sparse_indices = valid_idx[:take] + self.num_sink_tokens
                                    retrieved_depths = sorted_d[:take].float()
                                    if h == 0 and req_id == 0:
                                        log_depths.extend(retrieved_depths.tolist())
                    else:
                        raw_indices = idx_list[h]
                        if raw_indices.numel() > 0:
                            sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

                    if h == 0 and req_id == 0:
                        log_sink = sink_indices.numel()
                        log_local = local_indices.numel()
                        log_sparse = sparse_indices.numel()

                    full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

                    depth_map = torch.zeros(full_indices.shape[0], device=self.device)

                    if self.use_jungle and sparse_indices.numel() > 0:
                        effective_threshold = retrieved_depths.min().item()
                        sparse_set = set(sparse_indices.tolist())
                        depth_list = [effective_threshold if idx.item() in sparse_set else 0.0 for idx in full_indices]
                        depth_map = torch.tensor(depth_list, device=self.device)

                    k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
                    v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
                    avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
                    k_sel_centered = k_sel - avg_k
                    k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
                    is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

                    if self.use_jungle:
                        out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
                    else:
                        out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)

                    # --- VALIDATION BLOCK ---
                    gt_k = self.k_cache[layer_idx][req_id, kv_head, :curr_len, :]
                    gt_v = self.v_cache[layer_idx][req_id, kv_head, :curr_len, :]

                    gt_scores = torch.matmul(q_heads[h].unsqueeze(0), gt_k.transpose(0, 1)) / math.sqrt(dim)
                    gt_probs = torch.softmax(gt_scores, dim=-1)
                    gt_out = torch.matmul(gt_probs, gt_v).to(self.dtype)

                    diff = gt_out - out_h
                    error = diff.norm() / (gt_out.norm() + 1e-6)
                    self.approx_errors.append(error.item())
                    # --- END VALIDATION ---

                    head_outputs.append(out_h)
                hidden_states_list.append(torch.cat(head_outputs, dim=0))

        if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
            self.logger.log("total_tokens", self.current_len[0])
            self.logger.log("sink_tokens", log_sink)
            self.logger.log("local_tokens", log_local)
            self.logger.log("sparse_tokens", log_sparse)
            if self.use_jungle and log_depths:
                self.logger.log("avg_depth", np.mean(log_depths))
                self.logger.log("max_depth", np.max(log_depths))

        return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)

# ==========================================
# Part 6: Model Wrappers
# ==========================================

class LLMLayer:
    def __init__(self, layer_idx, hf_layer, device):
        self.layer_idx = layer_idx
        self.device = device
        self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
        self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
        self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
        self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
        self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
        self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
        self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
        self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
        self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
        self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
        self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

    def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
        residual = hidden_states
        hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
        bsz, q_len, _ = hidden_states.shape
        q = F.linear(hidden_states, self.wq)
        k = F.linear(hidden_states, self.wk)
        v = F.linear(hidden_states, self.wv)
        n_heads = attn_server.num_heads
        n_kv_heads = attn_server.num_kv_heads
        head_dim = attn_server.head_dim
        q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
        k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
        v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
        q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
        k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

        if is_prefill:
            for i in range(bsz):
                attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
            k_rep = repeat_kv(k, n_heads // n_kv_heads)
            v_rep = repeat_kv(v, n_heads // n_kv_heads)
            attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
            attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
        else:
            attn_output = attn_server.decode(q, k, v, self.layer_idx)

        hidden_states = F.linear(attn_output, self.wo)
        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
        up = F.linear(hidden_states, self.up_proj)
        gate = F.linear(hidden_states, self.gate_proj)
        down = F.linear(F.silu(gate) * up, self.down_proj)
        hidden_states = residual + down
        return hidden_states

class LLM:
    def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
        self.device = device
        self.config = LlamaConfig.from_pretrained(model_name)
        self.max_length = max_length
        print(f"Loading Model: {model_name}...")
        hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
        self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
        self.lm_head = hf_model.lm_head.weight.detach().to(device)
        self.norm_weight = hf_model.model.norm.weight.detach().to(device)
        self.norm_eps = hf_model.model.norm.variance_epsilon
        self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
        t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.cos_cache = emb.cos().to(torch.bfloat16)
        self.sin_cache = emb.sin().to(torch.bfloat16)
        self.layers = []
        for idx, layer in enumerate(hf_model.model.layers):
            self.layers.append(LLMLayer(idx, layer, device))
            hf_model.model.layers[idx] = None
            gc.collect()
        self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

        # Cleanup original HF model structure
        del hf_model
        cleanup_memory()

    def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
        self.attn_server.clear()
        self.attn_server.verbose = verbose
        bsz, seq_len = input_ids.shape
        position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)

        # Prefill
        hidden_states = F.embedding(input_ids, self.embed_tokens)
        for layer in self.layers:
            hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)

        generated = []
        curr_pos = seq_len
        logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
        next_token = topp_temperature_decode(logits, temperature)
        generated.append(next_token.item())

        # Decode
        start_time = time.time()
        for i in range(max_tokens):
            input_ids = next_token
            position_ids = torch.tensor([[curr_pos]], device=self.device)
            hidden_states = F.embedding(input_ids, self.embed_tokens)
            for layer in self.layers:
                hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
            self.attn_server.step()
            logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
            next_token = topp_temperature_decode(logits, temperature)
            generated.append(next_token.item())
            curr_pos += 1
            if next_token.item() in [128001, 128009]:
                break

        end_time = time.time()

        # Collect Stats
        duration = end_time - start_time
        tokens_gen = len(generated)
        tps = tokens_gen / duration if duration > 0 else 0

        stats = self.attn_server.logger.get_aggregated_stats()
        stats['tps'] = tps
        if self.attn_server.approx_errors:
            stats['error'] = np.mean(self.attn_server.approx_errors)
        else:
            stats['error'] = 0.0

        return generated, stats

# ==========================================
# Part 7: Experiment Harness
# ==========================================

if __name__ == "__main__":
    MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

    prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers such as David are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is 50. Ok now that the math is done, let's tell a story but start with the name of the teacher we mentioned. Once upon a time, "
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

    # Storage for experiment results
    results = collections.defaultdict(lambda: collections.defaultdict(list))

    def run_benchmark(method_name, use_jungle, run_id):
        print(f"\n🚀 STARTING {method_name} RUN {run_id+1}/5")
        cleanup_memory()

        try:
            # Init Model
            llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=use_jungle)

            # Generate
            out_ids, stats = llm.generate(input_ids, max_tokens=20, verbose=False)
            text = tokenizer.decode(out_ids)

            # Log Data
            results[method_name]['error'].append(stats['error'])
            results[method_name]['sparsity'].append(stats.get('sparsity_ratio', 0))
            results[method_name]['tps'].append(stats['tps'])

            print(f"   [Done] TPS: {stats['tps']:.2f} | Error: {stats['error']:.4f} | Sparse%: {stats.get('sparsity_ratio',0)*100:.1f}%")
            print(f"   Output: {text}")

            # Cleanup Object
            del llm
        except Exception as e:
            print(f"   [CRASH] Run {run_id} failed: {e}")

        cleanup_memory()

    # # --- RUN EXPERIMENTS ---
    # for i in range(5):
    #     run_benchmark("MagicPIG", use_jungle=False, run_id=i)

    # for i in range(5):
    #     run_benchmark("Jungle", use_jungle=True, run_id=i)

    # # --- PRINT FINAL REPORT ---
    # print("\n\n" + "="*80)
    # print(f"{'METHOD':<15} | {'ERROR (L2)':<20} | {'SPARSITY (%)':<20} | {'SPEED (TPS)':<20}")
    # print("="*80)

    # for method in ["MagicPIG", "Jungle"]:
    #     err = results[method]['error']
    #     spr = results[method]['sparsity']
    #     tps = results[method]['tps']

    #     if not err:
    #         print(f"{method:<15} | N/A")
    #         continue

    #     e_mean, e_std = np.mean(err), np.std(err)
    #     s_mean, s_std = np.mean(spr) * 100, np.std(spr) * 100
    #     t_mean, t_std = np.mean(tps), np.std(tps)

    #     print(f"{method:<15} | {e_mean:.4f} ± {e_std:.4f}     | {s_mean:.1f}% ± {s_std:.1f}%       | {t_mean:.2f} ± {t_std:.2f}")

    print("="*80)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# ==========================================
# Part 7: Proper "Needle in a Haystack" Harness
# ==========================================

def generate_niah_prompt(tokenizer, n_context=500, needle_depth=0.5):
    """
    Creates a 'Haystack' of repeated junk text and hides a 'Needle' at a specific depth.
    """
    # 1. The Needle
    needle = " The special passcode is 'MAGIC_JUNGLE_99'. "

    # 2. The Haystack (Filler)
    filler = " The quick brown fox jumps over the lazy dog." * 5 # ~50 tokens
    filler_tokens = tokenizer.encode(filler, add_special_tokens=False)

    # Calculate how much filler we need
    target_filler_tokens = n_context - len(tokenizer.encode(needle)) - 50 # Reserve space for query
    repeats = target_filler_tokens // len(filler_tokens)

    # 3. Construct Context
    full_tokens = filler_tokens * repeats

    # 4. Insert Needle at Depth
    insert_idx = int(len(full_tokens) * needle_depth)
    needle_tokens = tokenizer.encode(needle, add_special_tokens=False)
    full_tokens = full_tokens[:insert_idx] + needle_tokens + full_tokens[insert_idx:]

    # 5. Append Query
    query = " What is the special passcode? The passcode is"
    query_tokens = tokenizer.encode(query, add_special_tokens=False)
    final_tokens = full_tokens + query_tokens

    return torch.tensor([final_tokens], dtype=torch.long, device=DEVICE)

if __name__ == "__main__":
    MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # --- EXPERIMENT CONFIG ---
    N_CONTEXT = 2048  # Make this large enough to force Sparse Attention ( > local_window)
    NEEDLE_DEPTHS = [0.1, 0.5, 0.9] # Test beginning, middle, and end

    print(f"🔎 RUNNING NEEDLE-IN-A-HAYSTACK (Context: {N_CONTEXT} tokens)")

    def run_niah_benchmark(method_name, use_jungle):
        print(f"\n--- {method_name} Evaluation ---")

        for depth in NEEDLE_DEPTHS:
            cleanup_memory()
            try:
                # 1. Generate Prompt
                input_ids = generate_niah_prompt(tokenizer, N_CONTEXT, needle_depth=depth)

                # 2. Init Model
                llm = LLM(MODEL_NAME, K=10, L=150, max_length=N_CONTEXT+100, device=DEVICE, use_jungle=use_jungle)

                # 3. Generate Answer (we only need ~10 tokens for the passkey)
                out_ids, stats = llm.generate(input_ids, max_tokens=10, verbose=False)
                output_text = tokenizer.decode(out_ids) # Only decode new tokens

                # 4. Check Success
                success = "MAGIC_JUNGLE_99" in output_text
                status = "✅ FOUND" if success else f"❌ MISSED (Got: '{output_text.strip()}')"

                print(f"   Depth {int(depth*100)}%: {status} | Error: {stats['error']:.4f} | Sparse: {stats.get('sparsity_ratio',0)*100:.1f}%")

                del llm
            except Exception as e:
                print(f"   [CRASH] Depth {depth} failed: {e}")

    # Run Comparisons
    run_niah_benchmark("MagicPIG (LSH)", use_jungle=False)
    run_niah_benchmark("Jungle (SNIS)", use_jungle=True)

🔎 RUNNING NEEDLE-IN-A-HAYSTACK (Context: 2048 tokens)

--- MagicPIG (LSH) Evaluation ---


`torch_dtype` is deprecated! Use `dtype` instead!


Loading Model: meta-llama/Meta-Llama-3-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Depth 10%: ✅ FOUND | Error: 0.2287 | Sparse: 1.2%
Loading Model: meta-llama/Meta-Llama-3-8B-Instruct...


...

   Depth 50%: ✅ FOUND | Error: 0.2356 | Sparse: 1.0%
Loading Model: meta-llama/Meta-Llama-3-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   Depth 90%: ✅ FOUND | Error: 0.2434 | Sparse: 1.3%

--- Jungle (SNIS) Evaluation ---
Loading Model: meta-llama/Meta-Llama-3-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi

#     # 1. Calculate collision prob at the realized depth
#     p_collision = p_base.pow(retrieval_depths.float())

#     # 2. Probability of at least one collision across L trees
#     w = 1.0 - (1.0 - p_collision).pow(L)

#     # --- FIX: Numerical Stability ---
#     # Clamp probability to avoid log(0) without adding a large bias (1e-4)
#     log_w = torch.log(torch.clamp(w, min=1e-10))
#     # --------------------------------

#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05): # Removed hardcoded Jungle params

#         # [NEW] Track Approximation Error
#         self.approx_errors = []

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2
#         self.logging_layer = 15

#         # [MODIFIED] Enforce "Same Size Table" Rule
#         # We force Jungle to use the exact same dimensions as MagicPIG
#         self.jg_K_max = K
#         self.jg_L = L
#         self.jg_budget = jg_budget
#         self.jg_min_depth = 1

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (SNIS Mode)")
#             print(f"   - Strict Fairness Mode: Using K={self.jg_K_max}, L={self.jg_L} (Same as MagicPIG)")
#             # [MODIFIED] Use the synced K and L sizes
#             self.jg_projs = torch.randn(self.head_dim, self.jg_L * self.jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")
#             print(f"   - Config: K={K}, L={L}")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.approx_errors = [] # Clear errors
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#         # [NEW] Print Error Stats
#         if self.approx_errors:
#             avg_err = np.mean(self.approx_errors)
#             print(f"📉 APPROXIMATION ERROR (Relative L2): {avg_err:.5f}")
#             print(f"   (Lower is better)")
#             print("="*60 + "\n")

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     # [MODIFIED] Use correct reshape based on synced sizes
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     # [MODIFIED] Reshape using synced sizes
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)

#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         effective_threshold = retrieved_depths.min().item()
#                         sparse_set = set(sparse_indices.tolist())
#                         depth_list = [effective_threshold if idx.item() in sparse_set else 0.0 for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)

#                     # [NEW] --- VALIDATION BLOCK ---
#                     # 1. Grab FULL history for this head (Ground Truth)
#                     gt_k = self.k_cache[layer_idx][req_id, kv_head, :curr_len, :]
#                     gt_v = self.v_cache[layer_idx][req_id, kv_head, :curr_len, :]

#                     # 2. Compute Exact Dense Attention
#                     gt_scores = torch.matmul(q_heads[h].unsqueeze(0), gt_k.transpose(0, 1)) / math.sqrt(dim)
#                     gt_probs = torch.softmax(gt_scores, dim=-1)
#                     gt_out = torch.matmul(gt_probs, gt_v).to(self.dtype)

#                     # 3. Compute Relative Error
#                     diff = gt_out - out_h
#                     error = diff.norm() / (gt_out.norm() + 1e-6)
#                     self.approx_errors.append(error.item())
#                     # [NEW] --- END VALIDATION ---

#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers such as David are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is 50. Ok now that the math is done, let's tell a story but start with the name of the teacher we mentioned. Once upon a time, "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # # 1. RUN MAGICPIG
#     # print("\n\n" + "#"*40)
#     # print("RUNNING MAGICPIG (LSH)")
#     # print("#"*40)
#     # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#     out_jg = llm_jg.generate(input_ids, max_tokens=20, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi

#     # 1. Calculate collision prob at the realized depth
#     p_collision = p_base.pow(retrieval_depths.float())

#     # 2. Probability of at least one collision across L trees
#     w = 1.0 - (1.0 - p_collision).pow(L)

#     # --- FIX: Numerical Stability ---
#     # Clamp probability to avoid log(0) without adding a large bias (1e-4)
#     log_w = torch.log(torch.clamp(w, min=1e-10))
#     # --------------------------------

#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.approx_errors = []
#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2
#         self.logging_layer = 15

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#         if self.approx_errors:
#             avg_err = np.mean(self.approx_errors)
#             print(f"📉 APPROXIMATION ERROR (Relative L2): {avg_err:.4f}")
#             print(f"   (Lower is better. < 0.10 is usually negligible for PPL)")
#             print("="*60 + "\n")

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # --- CRITICAL FIX START (Probability Calculation) ---
#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)

#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         # 1. Identify the effective threshold used for this batch
#                         effective_threshold = retrieved_depths.min().item()

#                         # 2. Assign this threshold to ALL retrieved keys
#                         # We do NOT use the individual depths. We use the threshold that qualified them.
#                         sparse_set = set(sparse_indices.tolist())

#                         # If index is sparse, it gets threshold depth.
#                         # If it is exact (sink/local), it gets 0.0 (which results in p=1.0, w=1.0, log_w=0)
#                         depth_list = [effective_threshold if idx.item() in sparse_set else 0.0 for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)
#                     # --- CRITICAL FIX END ---

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)

#                     # 1. Grab FULL history for this head (Ground Truth)
#                     # We use :curr_len to get all tokens, ignoring sparsity
#                     gt_k = self.k_cache[layer_idx][req_id, kv_head, :curr_len, :]
#                     gt_v = self.v_cache[layer_idx][req_id, kv_head, :curr_len, :]

#                     # 2. Compute Exact Dense Attention
#                     # (1, dim) @ (dim, seq_len) -> (1, seq_len)
#                     gt_scores = torch.matmul(q_heads[h].unsqueeze(0), gt_k.transpose(0, 1)) / math.sqrt(dim)
#                     gt_probs = torch.softmax(gt_scores, dim=-1)
#                     gt_out = torch.matmul(gt_probs, gt_v).to(self.dtype)

#                     # 3. Compute Relative Error (L2 Norm of Difference / L2 Norm of Truth)
#                     # How far off was our sparse vector 'out_h' from the truth 'gt_out'?
#                     diff = gt_out - out_h
#                     error = diff.norm() / (gt_out.norm() + 1e-6)
#                     self.approx_errors.append(error.item())
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # ==========================================
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers such as David are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is 50. Ok now that the math is done, let's tell a story but start with the name of the teacher we mentioned. Once upon a time, "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # # 1. RUN MAGICPIG
#     # print("\n\n" + "#"*40)
#     # print("RUNNING MAGICPIG (LSH)")
#     # print("#"*40)
#     # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     out_jg = llm_jg.generate(input_ids, max_tokens=20, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

For Jungle: APPROXIMATION ERROR (Relative L2): 0.2991

For Magic Pig: APPROXIMATION ERROR (Relative L2): 0.4727